# MolMage on Colab

Turn a molecule (SMILES or SDF) into a **MolMage**, the 64 × 64 × 4 surface image used by [ImagenMol](https://anonymous.4open.science/r/ImagenMol-65A8/README.md), decode it back into a 3D mesh, and look at Blender snapshots of that mesh.

`SMILES / SDF → RDKit conformer → PyMOL surface → Open3D UV atlas → Blender bake → MolMage → mesh + snapshots`

Run the cells in order; a CPU runtime is enough. The code lives in the private [`molmage-colab`](https://github.com/Qian-Anton/molmage-colab) repository, which step 1 clones with a GitHub token stored as a Colab secret (see *Access* in the notes at the end), so each cell only sets a few options.

In [ ]:
#@title 1 · Setup (first run ≈ 3–5 min) { display-mode: "form" }
#@markdown Clones the helper package from its private GitHub repository, then builds a separate Python 3.10 environment with bpy 4.0.0, PyMOL, Open3D and RDKit (Colab's own Python is left untouched). The token comes from the Colab secret named in `TOKEN_SECRET`; see *Access* in the notes at the end.
REPO_URL = "https://github.com/Qian-Anton/molmage-colab.git"  #@param {type:"string"}
TOKEN_SECRET = "GITHUB_TOKEN"  #@param {type:"string"}
FORCE_REINSTALL = False  #@param {type:"boolean"}

import base64, os, subprocess, sys
from google.colab import userdata


def git(*args):
    """Run git; the token is sent to github.com as a header for this command only, never written to disk."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    try:
        token = userdata.get(TOKEN_SECRET)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException) as exc:
        print(f"No GitHub token: {exc}")
        token = ""
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}")
    return subprocess.run(["git", *args], env=env, capture_output=True, text=True)


repo = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(repo):
    git("-C", repo, "pull", "-q", "--ff-only")
else:
    clone = git("clone", "-q", "--depth", "1", REPO_URL, repo)
    if clone.returncode:
        raise RuntimeError(f"git clone failed. Check that the Colab secret {TOKEN_SECRET!r} holds a GitHub token "
                           f"that can read {REPO_URL}\n{clone.stderr.strip()}")
if repo not in sys.path:
    sys.path.insert(0, repo)
for module in [m for m in sys.modules if m.split(".")[0] == "molmage_colab"]:
    del sys.modules[module]  # use the freshly pulled code

import molmage_colab as mc
versions = mc.setup(force=FORCE_REINSTALL)

In [ ]:
#@title 2 · Molecule → MolMage { display-mode: "form" }
INPUT = "SMILES"  #@param ["SMILES", "SDF file", "MolMage file (.npy / .pt)"]
SMILES = "CC(=O)Oc1ccccc1C(=O)O"  #@param {type:"string"}
#@markdown `FILE` (SDF or MolMage input): `upload`, an example SDF, or a path such as `/content/my.sdf`. `INDEX` picks the record in a multi-molecule SDF or the sample in a batched `.pt`.
FILE = "upload"  #@param ["upload", "aspirin.sdf", "caffeine.sdf", "ibuprofen.sdf", "lidocaine.sdf", "acetaminophen.sdf"] {allow-input: true}
INDEX = 0  #@param {type:"integer"}
#@markdown ---
RESOLUTION = 64  #@param [64, 128] {type:"raw"}
SEED = 42  #@param {type:"integer"}
SKIP_FILTERS = False  #@param {type:"boolean"}

result = mc.run(INPUT, smiles=SMILES, file=FILE, index=INDEX,
                resolution=RESOLUTION, seed=SEED, skip_filters=SKIP_FILTERS)

In [ ]:
#@title 3 · Mesh snapshots (Blender) { display-mode: "form" }
#@markdown Renders the mesh decoded from the MolMage with the official preview settings; 0° is the view used in `_mmg.png`.
VIEWS = 4  #@param {type:"slider", min:1, max:8, step:1}

mc.show_snapshots(views=VIEWS)

In [ ]:
#@title 4 · Interactive 3D view { display-mode: "form" }
#@markdown Left: the conformer inside the PyMOL surface that was encoded. Right: the mesh decoded from the MolMage. `position` paints both with the MolMage x, y, z colours.
COLOR_BY = "patch"  #@param ["patch", "position"]

mc.show_mesh(color_by=COLOR_BY)

In [ ]:
#@title 5 · Download { display-mode: "form" }
WHAT = "latest result"  #@param ["latest result", "all results"]

mc.download(everything=WHAT == "all results")

### Notes

- **Access.** The helper repository is private, so step 1 needs a GitHub token. Create a [fine-grained token](https://github.com/settings/personal-access-tokens/new) for the `molmage-colab` repository with read-only *Contents* permission, then add it in Colab under 🔑 *Secrets* as `GITHUB_TOKEN` with *Notebook access* on. Step 1 passes it to git as an HTTP header for github.com only; it is not saved in the cloned folder.
- **Fidelity.** The steps call the functions used by `MolMage/scripts/encode_molmage.py` and `decode_molmage.py` with the same settings (Open3D UV atlas 1024 / gutter 10, Blender bake at 1024², `downsample_omg` to 64; decoding writes the official `_mmg.glb`, `_mmg.blend` and `_mmg.png`). Differences: failures are reported instead of skipped, the 20–30 s thread timeouts are not applied, and SMILES conformers use a fixed seed (`SEED = -1` restores the unseeded original).
- **Reproducibility.** Open3D's UV atlas is not deterministic, so re-encoding the same conformer changes the patch layout of the MolMage. The decoded shape stays the same to about 1 % of the box size.
- **Input range.** MolMage keeps molecules like its training set: 6–50 heavy atoms, ≤ 10 rotatable bonds, elements H C N O F P S Cl Br I, and no `+ - . / \` in the SMILES. The error message says which rule failed; `SKIP_FILTERS` runs the molecule anyway.
- **Hydrogens.** SMILES input is embedded with hydrogens, SDF input is read without them (as in the official scripts), so the same molecule gives different surfaces.
- **Environment.** bpy 4.0.0 comes from download.blender.org (PyPI no longer has it, and this code needs Blender < 4.1), PyMOL from `pymol-open-source-whl`, and `pyatlas` (only in the official Docker image) is replaced by a stub because the default Open3D UV method does not use it. Outputs are in `/content/molmage_outputs/<name>/`.